In [1]:
!pip install -q langchain-text-splitters langchain-huggingface langchain-chroma gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import gradio as gr
import json

In [3]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
DATASET_PATH = "/content/songs_dataset_50_final.json"

def load_dataset(path):
    with open(path, "r", encoding="utf-8") as file:
        dataset = json.load(file)
    return dataset

dataset = load_dataset(DATASET_PATH)

print(f"Loaded {len(dataset)} songs.")
dataset[0]

Loaded 50 songs.


{'id': 1,
 'song_name': 'Shape of You',
 'artist': 'Ed Sheeran',
 'genre': 'Pop',
 'mood': 'Romantic',
 'tempo': 'Medium',
 'language': 'English',
 'release_year': 2017,
 'description': 'This Pop song by Ed Sheeran delivers a romantic listening experience with a medium tempo. Released in 2017, it blends memorable melodies, polished production, and expressive vocals to create a distinctive atmosphere. The arrangement balances rhythm, harmony, and emotion, making it enjoyable from the first listen while revealing more details over repeated plays. Listeners often associate the track with moments such as road trips, evening drives, studying, relaxing, celebrations, workouts, or spending time with friends depending on its mood and energy. Its musical style fits naturally alongside other songs in the Pop genre and appeals to fans looking for similar sounds, instrumentation, and emotional themes. The lyrics explore ideas connected to romantic emotions, personal experiences, relationships, and

In [5]:
def create_documents(dataset):
    documents = []

    for song in dataset:
        text = f"""
Song Name: {song['song_name']}
Artist: {song['artist']}
Genre: {song['genre']}
Mood: {song['mood']}
Tempo: {song['tempo']}
Language: {song['language']}
Release Year: {song['release_year']}

Description:
{song['description']}
"""

        document = Document(
            page_content=text,
            metadata={
                "id": song["id"],
                "song_name": song["song_name"],
                "artist": song["artist"],
                "genre": song["genre"],
                "mood": song["mood"],
                "tempo": song["tempo"],
                "language": song["language"],
                "release_year": song["release_year"]
            }
        )

        documents.append(document)

    return documents

In [6]:
documents = create_documents(dataset)

print(f"Total Documents: {len(documents)}")
print(documents[0])

Total Documents: 50
page_content='
Song Name: Shape of You
Artist: Ed Sheeran
Genre: Pop
Mood: Romantic
Tempo: Medium
Language: English
Release Year: 2017

Description:
This Pop song by Ed Sheeran delivers a romantic listening experience with a medium tempo. Released in 2017, it blends memorable melodies, polished production, and expressive vocals to create a distinctive atmosphere. The arrangement balances rhythm, harmony, and emotion, making it enjoyable from the first listen while revealing more details over repeated plays. Listeners often associate the track with moments such as road trips, evening drives, studying, relaxing, celebrations, workouts, or spending time with friends depending on its mood and energy. Its musical style fits naturally alongside other songs in the Pop genre and appeals to fans looking for similar sounds, instrumentation, and emotional themes. The lyrics explore ideas connected to romantic emotions, personal experiences, relationships, and self-expression w

In [7]:
def create_vector_db(documents):
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory="./songs_db"
    )

    return vectorstore
vectorstore = create_vector_db(documents)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [9]:
def recommend_songs(query,
                    genre="Any",
                    language="Any",
                    mood="Any",
                    release_year=0):

    results = vectorstore.similarity_search_with_score(
        query=query,
        k=10
    )

    recommendations = []

    for doc, score in results:

        # Genre Filter
        if genre != "Any" and doc.metadata["genre"] != genre:
            continue

        # Language Filter
        if language != "Any" and doc.metadata["language"] != language:
            continue

        # Mood Filter
        if mood != "Any" and doc.metadata["mood"] != mood:
            continue

        # Release Year Filter
        if doc.metadata["release_year"] < release_year:
            continue

        recommendations.append({
            "Song Name": doc.metadata["song_name"],
            "Artist": doc.metadata["artist"],
            "Genre": doc.metadata["genre"],
            "Mood": doc.metadata["mood"],
            "Language": doc.metadata["language"],
            "Release Year": doc.metadata["release_year"],
            "Similarity Score": round(score, 4)
        })

    return recommendations[:5]

In [10]:
def recommend(query, genre, language, mood, release_year):
    results = recommend_songs(
        query=query,
        genre=genre,
        language=language,
        mood=mood,
        release_year=release_year
    )

    if not results:
        return "No songs found matching your criteria."

    output = ""

    for i, song in enumerate(results, 1):
        output += f"""🎵 Recommendation {i}

Song Name       : {song['Song Name']}
Artist          : {song['Artist']}
Genre           : {song['Genre']}
Mood            : {song['Mood']}
Language        : {song['Language']}
Release Year    : {song['Release Year']}
Similarity Score: {song['Similarity Score']}

{'-'*50}
"""

    return output


genres = ["Any"] + sorted({song["genre"] for song in dataset})
languages = ["Any"] + sorted({song["language"] for song in dataset})
moods = ["Any"] + sorted({song["mood"] for song in dataset})


with gr.Blocks(title="Music Recommendation System") as demo:

    gr.Markdown("# 🎵 Music Recommendation System")
    gr.Markdown("Enter a music preference and optionally apply filters.")

    query = gr.Textbox(
        label="Music Preference",
        placeholder="e.g. Suggest relaxing songs for studying"
    )

    genre = gr.Dropdown(
        choices=genres,
        value="Any",
        label="Genre"
    )

    language = gr.Dropdown(
        choices=languages,
        value="Any",
        label="Language"
    )

    mood = gr.Dropdown(
        choices=moods,
        value="Any",
        label="Mood"
    )

    release_year = gr.Slider(
        minimum=min(song["release_year"] for song in dataset),
        maximum=max(song["release_year"] for song in dataset),
        value=min(song["release_year"] for song in dataset),
        step=1,
        label="Released After"
    )

    output = gr.Textbox(
        label="Recommendations",
        lines=20
    )

    recommend_button = gr.Button("Recommend Songs")

    recommend_button.click(
        fn=recommend,
        inputs=[
            query,
            genre,
            language,
            mood,
            release_year
        ],
        outputs=output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39aaca27cea5668cde.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
